In [1]:
# 필요한 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [14]:
# 최적화 결과 데이터 불러오기

radius_km = 5
p = 16

assignment_df = pd.read_csv(f"results/assignment_r{radius_km}_p{p}.csv")
selected_sites = pd.read_csv(f"results/selected_sites_r{radius_km}_p{p}.csv")

In [25]:
res = {
    "assignment_df": assignment_df
}

In [21]:
assignment_df.head()
assignment_df.tail()

,candidate_index,candidate_name,demand_index,demand_value,distance_m
2743,422,사당중학교,2630,23.0,3367.161232
2744,422,사당중학교,2669,109.0,2479.155964
2745,422,사당중학교,2690,122.0,4653.311163
2746,422,사당중학교,2737,111.0,3304.344547
2747,422,사당중학교,2761,45.0,2347.180402


In [15]:
# df_candidate 불러오기
df_candidate = pd.read_csv("data/final_candidates_v1.csv", encoding="cp949")

In [22]:
df_candidate.head()

,학교명,학교급구분,설립형태,소재지도로명주소,시도교육청명,교육지원청명,위도,경도,지역_x,시도교육청,...,제외여부,제외사유,교사대지,체육장,계,부속토지,합 계,공동사용여부,공동사용학교명,설립유형
0,대진여자고등학교,고등학교,사립,서울특별시 노원구 공릉로 438,서울특별시교육청,서울특별시북부교육지원청,37.646174,127.067197,노원구,서울특별시교육청,...,N,NaN,15802.0,2091.0,17893.0,0.0,17893.0,×,NaN,단설
1,예일디자인고등학교,고등학교,사립,서울특별시 은평구 연서로 117,서울특별시교육청,서울특별시서부교육지원청,37.610946,126.914793,은평구,서울특별시교육청,...,N,NaN,0.0,6459.0,0.0,0.0,0.0,○,예일여자고등학교,단설
2,대일관광고등학교,고등학교,사립,서울특별시 양천구 신정이펜1로 11,서울특별시교육청,서울특별시강서양천교육지원청,37.511414,126.834907,양천구,서울특별시교육청,...,N,NaN,9753.0,3898.0,13651.0,0.0,13651.0,×,NaN,단설
3,강동고등학교,고등학교,사립,서울특별시 강동구 구천면로 572,서울특별시교육청,서울특별시강동송파교육지원청,37.549917,127.160668,강동구,서울특별시교육청,...,N,NaN,2177.0,5261.0,7438.0,904.0,8342.0,×,NaN,단설
4,염광고등학교,고등학교,사립,서울특별시 노원구 월계로45가길 9,서울특별시교육청,서울특별시북부교육지원청,37.630284,127.049265,노원구,서울특별시교육청,...,N,NaN,0.0,4800.0,0.0,0.0,0.0,○,염광여자메디텍고등학교,단설


In [26]:
# 큐잉 모델 M/D/s
# =========================
# 0. 필요한 라이브러리
# =========================
import math
import numpy as np
import pandas as pd


# =========================
# 1. M/M/s 대기확률(Erlang C)
# =========================
def erlang_c(lmbda: float, mu: float, s: int) -> float:
    """
    M/M/s의 대기확률 Pw 계산
    lmbda: arrival rate (vehicles/hour)
    mu: service rate per server (vehicles/hour)
    s: number of servers
    """
    if s <= 0:
        raise ValueError("s must be >= 1")

    rho = lmbda / (s * mu) if s * mu > 0 else np.inf
    if rho >= 1:
        return 1.0

    a = lmbda / mu  # offered load

    sum_terms = sum((a ** n) / math.factorial(n) for n in range(s))
    last_term = (a ** s) / (math.factorial(s) * (1 - rho))
    p0 = 1.0 / (sum_terms + last_term)
    pw = last_term * p0
    return pw


# =========================
# 2. M/D/s 근사 지표 계산
#    - M/M/s 대기시간의 0.5배로 근사
# =========================
def approx_mds_metrics(lmbda: float, service_time_min: float, s: int) -> dict:
    """
    M/D/s 근사:
    Wq(M/D/s) ≈ 0.5 * Wq(M/M/s)

    반환:
    - rho
    - Pw
    - Wq_hr, W_hr
    - Lq, L
    - stable
    """
    mu = 60.0 / service_time_min  # vehicles/hour/server

    if s <= 0:
        return {
            "mu_per_hr": mu,
            "rho": np.nan,
            "Pw": np.nan,
            "Wq_hr": np.nan,
            "W_hr": np.nan,
            "Lq": np.nan,
            "L": np.nan,
            "stable": False,
        }

    rho = lmbda / (s * mu) if s * mu > 0 else np.inf
    if rho >= 1:
        return {
            "mu_per_hr": mu,
            "rho": rho,
            "Pw": 1.0,
            "Wq_hr": np.inf,
            "W_hr": np.inf,
            "Lq": np.inf,
            "L": np.inf,
            "stable": False,
        }

    # M/M/s 기준
    pw = erlang_c(lmbda, mu, s)
    wq_mms = pw / (s * mu - lmbda)  # hours

    # M/D/s 근사
    wq_mds = 0.5 * wq_mms
    w_mds = wq_mds + (1.0 / mu)

    lq = lmbda * wq_mds
    l_ = lmbda * w_mds

    return {
        "mu_per_hr": mu,
        "rho": rho,
        "Pw": pw,
        "Wq_hr": wq_mds,
        "W_hr": w_mds,
        "Lq": lq,
        "L": l_,
        "stable": True,
    }


# =========================
# 3. usable area, 서버 수 계산
# =========================
def add_server_info(
    df_candidate: pd.DataFrame,
    area_col: str = "체육장",
    utilization_ratio: float = 0.5,
    area_per_truck: float = 124.0,
    min_servers: int = 1,
) -> pd.DataFrame:
    """
    usable_area = 체육장 면적 * utilization_ratio
    servers = floor(usable_area / area_per_truck)
    """
    df = df_candidate.copy()

    df["usable_area"] = df[area_col] * utilization_ratio
    df["servers"] = np.floor(df["usable_area"] / area_per_truck).astype(int)
    df["servers"] = df["servers"].clip(lower=min_servers)

    return df


# =========================
# 4. assignment_df -> 학교별 수요 집계
# =========================
def summarize_assignment(
    assignment_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    assignment_df에서 학교별로
    - 담당 수요지 목록
    - 총수요
    - 수요지 수
    - 평균/최대 거리
    를 계산
    """
    if assignment_df is None or len(assignment_df) == 0:
        return pd.DataFrame(columns=[
            "candidate_index", "candidate_name", "assigned_demand_indices",
            "num_demand_points", "total_demand", "avg_distance_m", "max_distance_m"
        ])

    summary = (
        assignment_df.groupby(["candidate_index", "candidate_name"])
        .agg(
            assigned_demand_indices=("demand_index", list),
            num_demand_points=("demand_index", "count"),
            total_demand=("demand_value", "sum"),
            avg_distance_m=("distance_m", "mean"),
            max_distance_m=("distance_m", "max"),
        )
        .reset_index()
    )
    return summary


# # =========================
# # 5. 학교별 큐잉 입력/출력 계산
# # =========================
# def build_queue_table(
#     result: dict,
#     df_candidate: pd.DataFrame,
#     area_col: str = "체육장",
#     school_name_col: str = "학교명",
#     utilization_ratio: float = 0.5,
#     area_per_truck: float = 124.0,
#     vehicle_capacity: float = 24.0,   # 차량 1대당 최대 24가구
#     inbound_window_hr: float = 1.5,   # 차량이 몰려 들어오는 시간창
#     service_time_min: float = 15.0,   # 차량 1대 처리시간(분)
# ) -> pd.DataFrame:
#     """
#     Gurobi 결과(result["assignment_df"])를 이용해서
#     학교별 M/D/s 분석 테이블 생성
#     """
#     assignment_df = result.get("assignment_df", None)
#     if assignment_df is None or len(assignment_df) == 0:
#         return pd.DataFrame()

#     # 1) 학교별 총수요 집계
#     demand_summary = summarize_assignment(assignment_df)

#     # 2) 서버 수 계산용 candidate 정보 준비
#     candidate_with_server = add_server_info(
#         df_candidate=df_candidate,
#         area_col=area_col,
#         utilization_ratio=utilization_ratio,
#         area_per_truck=area_per_truck,
#     ).reset_index().rename(columns={"index": "candidate_index"})

#     # 병합
#     cols_to_use = ["candidate_index", school_name_col, area_col, "usable_area", "servers"]
#     queue_df = demand_summary.merge(
#         candidate_with_server[cols_to_use],
#         left_on=["candidate_index", "candidate_name"],
#         right_on=["candidate_index", school_name_col],
#         how="left"
#     )

#     # 3) 학교별 차량 수 / arrival rate 계산
#     # total_demand = 학교가 맡는 총 주문(가구) 수라고 해석
#     # total_vehicle_trips = 필요 차량 대수(또는 차량 투입 횟수)
#     queue_df["total_vehicle_trips"] = queue_df["total_demand"] / vehicle_capacity
#     queue_df["lambda_per_hr"] = queue_df["total_vehicle_trips"] / inbound_window_hr

#     # 4) 학교별 M/D/s 지표 계산
#     metrics = []
#     for _, row in queue_df.iterrows():
#         metric = approx_mds_metrics(
#             lmbda=row["lambda_per_hr"],
#             service_time_min=service_time_min,
#             s=int(row["servers"])
#         )
#         metrics.append(metric)

#     metrics_df = pd.DataFrame(metrics)
#     queue_df = pd.concat([queue_df.reset_index(drop=True), metrics_df.reset_index(drop=True)], axis=1)

#     # 보기 좋게 단위 컬럼 추가
#     queue_df["Wq_min"] = queue_df["Wq_hr"] * 60
#     queue_df["W_min"] = queue_df["W_hr"] * 60

#     # 정리
#     final_cols = [
#         "candidate_index",
#         "candidate_name",
#         school_name_col,
#         area_col,
#         "usable_area",
#         "servers",
#         "num_demand_points",
#         "assigned_demand_indices",
#         "total_demand",
#         "avg_distance_m",
#         "max_distance_m",
#         "total_vehicle_trips",
#         "lambda_per_hr",
#         "mu_per_hr",
#         "rho",
#         "Pw",
#         "Wq_min",
#         "W_min",
#         "Lq",
#         "L",
#         "stable",
#     ]

#     # 중복 학교명 컬럼 처리
#     final_cols = [c for c in final_cols if c in queue_df.columns]
#     queue_df = queue_df[final_cols].sort_values("rho", ascending=False).reset_index(drop=True)

#     return queue_df

def build_queue_table(
    result: dict,
    df_candidate: pd.DataFrame,
    area_col: str = "체육장",
    school_name_col: str = "학교명",
    utilization_ratio: float = 0.5,
    area_per_truck: float = 124.0,
    vehicle_capacity: float = 24.0,
    inbound_window_hr: float = 1.5,
    service_time_min: float = 15.0,
) -> pd.DataFrame:
    
    # 0) 입력 체크
    if not isinstance(result, dict):
        raise TypeError(f"result must be dict, got {type(result)}")

    if "assignment_df" not in result:
        raise KeyError("result does not contain 'assignment_df'")

    assignment_df = result["assignment_df"]

    if assignment_df is None or len(assignment_df) == 0:
        print("assignment_df is empty")
        return pd.DataFrame()

    if area_col not in df_candidate.columns:
        raise KeyError(f"'{area_col}' not found in df_candidate columns: {df_candidate.columns.tolist()}")

    if school_name_col not in df_candidate.columns:
        raise KeyError(f"'{school_name_col}' not found in df_candidate columns: {df_candidate.columns.tolist()}")

    required_cols = ["candidate_index", "candidate_name", "demand_index", "demand_value", "distance_m"]
    missing_cols = [c for c in required_cols if c not in assignment_df.columns]
    if missing_cols:
        raise KeyError(f"assignment_df missing columns: {missing_cols}")

    # 1) 학교별 총수요 집계
    demand_summary = summarize_assignment(assignment_df)
    print("demand_summary shape:", demand_summary.shape)

    # 2) candidate 정보 준비
    candidate_with_server = add_server_info(
        df_candidate=df_candidate,
        area_col=area_col,
        utilization_ratio=utilization_ratio,
        area_per_truck=area_per_truck,
    ).reset_index().rename(columns={"index": "candidate_index"})

    print("candidate_with_server shape:", candidate_with_server.shape)

    cols_to_use = ["candidate_index", school_name_col, area_col, "usable_area", "servers"]
    queue_df = demand_summary.merge(
        candidate_with_server[cols_to_use],
        left_on=["candidate_index", "candidate_name"],
        right_on=["candidate_index", school_name_col],
        how="left"
    )

    print("queue_df after merge shape:", queue_df.shape)
    print(queue_df.head())

    # 3) 학교별 차량 수 / arrival rate 계산
    queue_df["total_vehicle_trips"] = queue_df["total_demand"] / vehicle_capacity
    queue_df["lambda_per_hr"] = queue_df["total_vehicle_trips"] / inbound_window_hr

    # 4) 학교별 M/D/s 지표 계산
    metrics = []
    for _, row in queue_df.iterrows():
        metric = approx_mds_metrics(
            lmbda=row["lambda_per_hr"],
            service_time_min=service_time_min,
            s=int(row["servers"])
        )
        metrics.append(metric)

    metrics_df = pd.DataFrame(metrics)
    queue_df = pd.concat(
        [queue_df.reset_index(drop=True), metrics_df.reset_index(drop=True)],
        axis=1
    )

    queue_df["Wq_min"] = queue_df["Wq_hr"] * 60
    queue_df["W_min"] = queue_df["W_hr"] * 60

    return queue_df

In [27]:
# radius_km = 5
# p = 15

# res = assignment_df

# queue_df = build_queue_table(
#     result=res,
#     df_candidate=df_candidate,
#     area_col="체육장",
#     school_name_col="학교명",
#     utilization_ratio=0.5,   # 체육장 50% 활용
#     area_per_truck=124.0,    # 11톤 트럭 기준 1대당 필요 면적
#     vehicle_capacity=24.0,   # 차량 1대당 최대 24가구
#     inbound_window_hr=1.5,   # 1.5시간 동안 차량 유입
#     service_time_min=15.0    # 차량 1대 처리시간 15분
# )

# print(queue_df)
radius_km = 5
p = 16

queue_df = build_queue_table(
    result=res,
    df_candidate=df_candidate,
    area_col="체육장",
    school_name_col="학교명",
    utilization_ratio=0.5,
    area_per_truck=124.0,
    vehicle_capacity=24.0,
    inbound_window_hr=1.5,
    service_time_min=15.0
)

print(queue_df.head())

demand_summary shape: (16, 7)


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer.Replace or remove non-finite values or cast to an integer typethat supports these values (e.g. 'Int64')